### Imports 

In [ ]:
from pathlib import Path
import geopandas as gpd
import fiona
import pandas as pd
import rasterio
from rasterio.features import rasterize
import numpy as np
from rasterio.enums import MergeAlg
from rasterio.transform import xy
import re

import sys, pathlib, importlib

sys.path.append("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
import Robyn_catchment_analysis
importlib.reload(Robyn_catchment_analysis)  # optional if you edit the file

print(Robyn_catchment_analysis.__file__)  # should show the path above



### Set base / output paths and read in files

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
basins_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_basins_100")


In [ ]:
output_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info")
# output_dir.mkdir(parents=True, exist_ok=True)  # create if missing (and parents)

In [ ]:
# drainage_tif = "/Users/robynhaggis/Documents/Geospatial_analysis/drainage_direction_100.tif"   # r.watershed drainage-direction raster (EPSG:3448)

drainage_tif = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/drainage_direction_100.tif"


In [ ]:
# test_basin_path = basins_dir / "basin_4159_2625.gpkg"   # <- a file, not the folder

test_basin_path = base_path / "dphil_paper_2/processed_data/upstream_basins_100/basin_4159_2625.gpkg"


layers = fiona.listlayers(test_basin_path)
print(layers)  




In [ ]:
test_basin = gpd.read_file(test_basin_path, layer=layers[0])

In [ ]:
land_use = gpd.read_file(base_path /"dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)
land_use.plot()

### Intersection of upstream catchments and land use 

#### Test run with one file

In [ ]:
test_intersection = test_land_use_hydrobasins_intersection = gpd.overlay(land_use, test_basin, how='intersection')

In [ ]:
test_intersection

In [ ]:
test_output_upstream_landuse_intersection = base_path / "dphil_paper_2/processed_data/upstream_catchment_info/test_upstream_landuse_intersection.shp"
test_intersection.to_file(test_output_upstream_landuse_intersection)

In [ ]:
test_intersection_saved = gpd.read_file(test_output_upstream_landuse_intersection)
display(test_intersection_saved.head())

#### Test areas 

In [ ]:
catchment_area_m2 = test_intersection_saved.geometry.area.sum()
catchment_area_km2 = catchment_area_m2 / 1_000_000
catchment_area_m2
catchment_area_km2

In [ ]:
test_intersection_saved["area_m2"] = test_intersection_saved.geometry.area
area_by_class = (
    test_intersection_saved.groupby("Classify", as_index=False)["area_m2"]
    .sum()
    .sort_values("area_m2", ascending=False)
)
area_by_class["area_km2"] = area_by_class["area_m2"] / 1_000_000

# Add km² and % of catchment
area_by_class["area_km2"] = area_by_class["area_m2"] / 1_000_000
area_by_class["pct_of_catchment"] = (
    100 * area_by_class["area_m2"] / catchment_area_m2 if catchment_area_m2 > 0 else 0.0
)

In [ ]:
area_by_class

In [ ]:
# ----------------------------------------------------------------------------
# Apply the function to each row and "explode" the dictionary so each row has a single key-value pair.
test_intersection_saved['frac_dict'] = test_intersection_saved.apply(Robyn_catchment_analysis.calculate_fractional_areas, axis=1)


In [ ]:
test_expanded_rows = []
for idx, row in test_intersection_saved.iterrows():
    for key, value in row['frac_dict'].items():
        new_row = row.copy()
        new_row['LandUseCategory'] = key
        new_row['Area'] = value
        test_expanded_rows.append(new_row)
        
test_expanded_gdf = gpd.GeoDataFrame(test_expanded_rows, crs=test_intersection_saved.crs)
test_expanded_gdf

In [ ]:
# # Calculate the forest flood equivalent area (i.e. the area that contributes to flood reduction)
# test_forest_flood_equivalent_gdf = test_expanded_gdf[test_expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes']
# test_forest_flood_equivalent_area = (
#     test_forest_flood_equivalent_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'forest_flood_equivalent_area'})
# )

# Filter to your forest-flood-equivalent rows
test_forest_flood_equivalent_gdf = test_expanded_gdf[
    test_expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes'
]
# test_forest_flood_equivalent_area['forest_flood_equivalent_area_km2'] = (
#     test_forest_flood_equivalent_area['forest_flood_equivalent_area_m2'] / 1_000_000
# )

test_afforestable_agri_gdf = test_expanded_gdf[
    test_expanded_gdf['LandUseCategory'] == 'afforestable_including_agriculture'
]

test_forest_flood_equivalent_gdf

# # Calculate the afforestable area (including the agricultural component)
# test_afforestable_agri_gdf = test_expanded_gdf[expanded_gdf['LandUseCategory'] == 'afforestable_including_agriculture']
# afforestable_agri_area = (
#     afforestable_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'afforestable_including_agriculture_area'})
# )


#### Loop over all the catchments

In [ ]:
# Intersect land use with the upstream catchments
# Find out upstream catchment area in km2
# Find out % of forest area and km2 forest
gpkg_paths = sorted(basins_dir.glob("*.gpkg"))


In [ ]:
out_rows = []

for p in gpkg_paths:
    # read the (only) layer from each GPKG
    layer = fiona.listlayers(p)[0]
    catch = gpd.read_file(p, layer=layer).to_crs(land_use.crs)


#### Calculate forest area and afforestable area in a catchment

In [ ]:
proportion_forest = {
    'Open dry forest - Short': 1,
    'Quarry': 0,
    'Fields and Secondary Forest': 0.5,
    'Urban': 0,
}


In [ ]:
land_use['Classify'].unique()

In [ ]:
# 1) Print one per line
for cls in sorted(land_use['Classify'].dropna().unique()):
    print(cls)

In [ ]:
forest_flood_equivalent_column = {
    'Bamboo': 1,
    'Bamboo and Fields': 0.5,                  # 50% Ag, 50% Bamboo (if you consider bamboo here)
    'Bamboo and Secondary Forest': 1,        # 50% Forest, 50% Ag (if you want some ag fraction)
    'Bare Rock': 0,
    'Bauxite Extraction': 0,
    'Buildings and other infrastructures': 0,
    'Closed broadleaved forest (Primary Forest)': 1,
    'Disturbed broadleaved forest (Secondary Forest)': 1,
    'Fields  and Bamboo': 0.5,                  # 50% Ag, 50% Bamboo
    'Fields and Secondary Forest': 0.5,        # 50% Ag, 50% Forest
    'Fields or Secondary Forest/Pine Plantation': 0.5,  # 50% Ag, 50% Secondary Forest/Pine
    'Fields: Bare Land': 0,
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 0,
    'Fields: Pasture,Human disturbed, grassland': 0,
    'Hardwood Plantation: Euculytus': 1,
    'Hardwood Plantation: Mahoe': 1,
    'Hardwood Plantation: Mahogany': 1, 
    'Hardwood Plantation: Mixed': 1,
    'Herbaceous Wetland': 0,
    'Mangrove Forest': 0,
    'Open dry forest - Short': 1,
    'Open dry forest - Tall (Woodland/Savanna)': 1,
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 1,
    'Quarry': 0,
    'Secondary Forest': 1,
    'Swamp Forest': 0,
    'Water Body': 0,
}


In [ ]:
afforestable_column = {
    'Bamboo': 0,
    'Bamboo and Fields': 0.5,                  # 50% Ag, 50% Bamboo (if you consider bamboo here)
    'Bamboo and Secondary Forest': 0,        # 50% Forest, 50% Ag (if you want some ag fraction)
    'Bare Rock': 0,
    'Bauxite Extraction': 1,
    'Buildings and other infrastructures': 0,
    'Closed broadleaved forest (Primary Forest)': 0,
    'Disturbed broadleaved forest (Secondary Forest)': 0,
    'Fields  and Bamboo': 0.5,                  # 50% Ag, 50% Bamboo
    'Fields and Secondary Forest': 0.5,        # 50% Ag, 50% Forest
    'Fields or Secondary Forest/Pine Plantation': 0.5,  # 50% Ag, 50% Secondary Forest/Pine
    'Fields: Bare Land': 1,
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 1,
    'Fields: Pasture,Human disturbed, grassland': 1,
    'Hardwood Plantation: Euculytus': 0,
    'Hardwood Plantation: Mahoe': 0,
    'Hardwood Plantation: Mahogany': 0, 
    'Hardwood Plantation: Mixed': 0,
    'Herbaceous Wetland': 0,
    'Mangrove Forest': 0,
    'Open dry forest - Short': 0,
    'Open dry forest - Tall (Woodland/Savanna)': 0,
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 0,
    'Quarry': 1,
    'Secondary Forest': 0,
    'Swamp Forest': 0,
    'Water Body': 0,
}




In [ ]:
land_use = land_use.copy()

land_use["forest_flood_equivalent_values"] = (
    land_use["Classify"].map(forest_flood_equivalent_column).fillna(0).astype(float)
)

land_use["afforestable_values"] = (
    land_use["Classify"].map(afforestable_column).fillna(0).astype(float)
)

unmapped_ffe = sorted(set(land_use["Classify"].dropna()) - set(forest_flood_equivalent_column))
unmapped_aff = sorted(set(land_use["Classify"].dropna()) - set(afforestable_column))
print("Unmapped (FFE):", unmapped_ffe)
print("Unmapped (AFF):", unmapped_aff)

# quick sanity check
land_use

out_gpkg = Path("/Users/robynhaggis/Documents/Geospatial_analysis/land_use_forest_and_afforestable.gpkg")

# if you only want specific columns, subset before saving
# land_use = land_use[["Classify", "forest_flood_equivalent_values", "afforestable_values", "geometry"]]

land_use.to_file(out_gpkg, layer="land_use", driver="GPKG")
print(f"Saved to {out_gpkg}")

In [ ]:
ref_raster = "/Users/robynhaggis/Documents/Geospatial_analysis/dem_filled.tif"  # reference grid (CRS, extent, res)
out_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info")


out_existing_forest_raster = out_dir / "forest_flood_equivalent_values.tif"
out_afforestable_areas = out_dir / "afforestable_values.tif"

In [ ]:
# Ensure the two columns exist and are numeric
land_use = land_use.copy()
land_use["forest_flood_equivalent_values"] = pd.to_numeric(
    land_use["forest_flood_equivalent_values"], errors="coerce"
).fillna(0)
land_use["afforestable_values"] = pd.to_numeric(
    land_use["afforestable_values"], errors="coerce"
).fillna(0)

nodata = -9999.0

with rasterio.open(ref_raster) as ref:
    profile = ref.profile
    transform = ref.transform
    shape = (ref.height, ref.width)
    crs = ref.crs

land_use = land_use.to_crs(crs)

ffe_shapes = list(zip(land_use.geometry, land_use["forest_flood_equivalent_values"]))
aff_shapes = list(zip(land_use.geometry, land_use["afforestable_values"]))

ffe_arr = rasterize(ffe_shapes, out_shape=shape, transform=transform,
                    fill=nodata, dtype="float32", all_touched=False, merge_alg=MergeAlg.replace)
aff_arr = rasterize(aff_shapes, out_shape=shape, transform=transform,
                    fill=nodata, dtype="float32", all_touched=False, merge_alg=MergeAlg.replace)

profile.update(dtype="float32", count=1, compress="lzw", nodata=nodata)
with rasterio.open(out_existing_forest_raster, "w", **profile) as dst: dst.write(ffe_arr, 1)
with rasterio.open(out_afforestable_areas, "w", **profile) as dst: dst.write(aff_arr, 1)

In [ ]:
print(f"Wrote:\n- {out_existing_forest_raster}\n- {out_afforestable_areas}")

In [ ]:
print("FFE uniques:", np.unique(ffe_arr[ffe_arr != nodata]))
print("AFF uniques:", np.unique(aff_arr[aff_arr != nodata]))
diff = np.where((ffe_arr==nodata)|(aff_arr==nodata), nodata, ffe_arr - aff_arr)
print("Diff min/max:", diff[diff != nodata].min(), diff[diff != nodata].max())

In [ ]:
aff_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/afforestable_values.tif")
ffe_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/forest_flood_equivalent_values.tif")
out_points = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/aff_points_with_ffe.geoparquet")

with rasterio.open(aff_path) as aff, rasterio.open(ffe_path) as ffe:
    # sanity: same grid
    assert (aff.width, aff.height) == (ffe.width, ffe.height)
    assert aff.transform == ffe.transform
    assert aff.crs == ffe.crs

    aff_arr = aff.read(1)
    ffe_arr = ffe.read(1)
    nodata = aff.nodata

    # choose which pixels to make into points
    mask = aff_arr != nodata
    # (optional) only where afforestable > 0
    # mask &= aff_arr > 0

    rows, cols = np.where(mask)
    xs, ys = xy(aff.transform, rows, cols, offset="center")

    gdf = gpd.GeoDataFrame(
        {
            # keep grid indices for joins later (match your naming)
            "dem_i": rows,   # row index
            "dem_j": cols,   # col index
            "afforestable": aff_arr[rows, cols].astype("float32"),
            "existing_forest": ffe_arr[rows, cols].astype("float32"),
        },
        geometry=gpd.points_from_xy(xs, ys),
        crs=aff.crs
    )

# save as GeoParquet (or GPKG if you prefer)
gdf.to_parquet(out_points, index=False)
print(f"Wrote {out_points} with {len(gdf):,} points")
gdf.to_file(out_points.with_suffix(".gpkg"), layer="aff_points_with_ffe", driver="GPKG")

In [ ]:


# Paths
points_gpkg = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/aff_points_with_ffe.gpkg")

# 1) Load points (one per pixel) and the test basin
pts = gpd.read_file(points_gpkg, layer="aff_points_with_ffe")
test_basin = gpd.read_file(test_basin_path, layer=layers[0])   # you already have test_basin_path & layers

# Align CRS
if test_basin.crs != pts.crs:
    test_basin = test_basin.to_crs(pts.crs)

# 2) Pixel area from the raster (so we don’t hardcode 900)
with rasterio.open(aff_path) as src:
    px_area_m2 = abs(src.transform.a * src.transform.e)  # e.g., 30 * 30 = 900

# 3) Keep only points inside the basin
#    (use 'intersects' to include any boundary-touching points)
pts_in = gpd.sjoin(pts, test_basin[["geometry"]], how="inner", predicate="intersects")

# Ensure numeric (should already be float32)
pts_in["afforestable"]     = pd.to_numeric(pts_in["afforestable"], errors="coerce").fillna(0)
pts_in["existing_forest"]  = pd.to_numeric(pts_in["existing_forest"], errors="coerce").fillna(0)

# 4) Effective area = sum of per-pixel fractions × pixel area
aff_m2 = float(pts_in["afforestable"].sum()) * px_area_m2
ffe_m2 = float(pts_in["existing_forest"].sum()) * px_area_m2

aff_km2 = aff_m2 / 1_000_000
ffe_km2 = ffe_m2 / 1_000_000

# 5) Optional: % of the catchment
catchment_area_m2 = float(test_basin.geometry.area.sum())
pct_aff = 100 * aff_m2 / catchment_area_m2 if catchment_area_m2 > 0 else 0.0
pct_ffe = 100 * ffe_m2 / catchment_area_m2 if catchment_area_m2 > 0 else 0.0

print(f"Pixel area: {px_area_m2:g} m²")
print(f"Afforestable area: {aff_m2:,.0f} m² ({aff_km2:.3f} km²) [{pct_aff:.1f}% of basin]")
print(f"Existing forest area: {ffe_m2:,.0f} m² ({ffe_km2:.3f} km²) [{pct_ffe:.1f}% of basin]")

In [ ]:
# --- paths ---
points_layer = "aff_points_with_ffe"
aff_raster  = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/afforestable_values.tif")
out_csv     = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/basin_aff_ffe_areas.csv")

# load points once
pts = gpd.read_file(points_gpkg, layer=points_layer)
pts["afforestable"]    = pd.to_numeric(pts["afforestable"], errors="coerce").fillna(0).astype("float32")
pts["existing_forest"] = pd.to_numeric(pts["existing_forest"], errors="coerce").fillna(0).astype("float32")

# pixel area (e.g., 30*30 = 900 m²)
with rasterio.open(aff_raster) as src:
    px_area_m2 = abs(src.transform.a * src.transform.e)

# CSV header (includes 'total future forest' fields)
cols = [
    "basin_file", "dem_i_file", "dem_j_file", "dem_i_attr", "dem_j_attr",
    "basin_area_m2", "basin_area_km2",
    "afforestable_m2", "afforestable_km2", "afforestable_pct",
    "existing_forest_m2", "existing_forest_km2", "existing_forest_pct",
    "total_future_forest_m2", "total_future_forest_km2", "total_future_forest_pct",
    "points_used",
]
if not out_csv.exists():
    pd.DataFrame(columns=cols).to_csv(out_csv, index=False)

def parse_ids(stem: str):
    m = re.match(r"basin_(\d+)_(\d+)$", stem)  # matches basin_<i>_<j>
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

for basin_path in sorted(basins_dir.glob("*.gpkg")):
    layers = fiona.listlayers(basin_path)
    if not layers:
        print(f"Skip {basin_path.name}: no layers"); continue

    basin = gpd.read_file(basin_path, layer=layers[0])
    if basin.crs != pts.crs:
        basin = basin.to_crs(pts.crs)

    basin_area_m2 = float(basin.geometry.area.sum())
    basin_area_km2 = basin_area_m2 / 1_000_000

    # union geometry (Shapely 2)
    geom = basin.union_all()

    # prefilter via sindex, then precise filter
    idx = list(pts.sindex.query(geom, predicate="intersects"))
    cand = pts.iloc[idx]
    pts_in = cand[cand.intersects(geom)]

    # sums → areas
    aff_m2 = float(pts_in["afforestable"].sum()) * px_area_m2
    ffe_m2 = float(pts_in["existing_forest"].sum()) * px_area_m2
    total_future_forest_m2  = aff_m2 + ffe_m2

    # km² + percentages
    aff_km2 = aff_m2 / 1_000_000
    ffe_km2 = ffe_m2 / 1_000_000
    total_future_forest_km2 = total_future_forest_m2 / 1_000_000

    pct_aff   = 100 * aff_m2 / basin_area_m2 if basin_area_m2 > 0 else 0.0
    pct_ffe   = 100 * ffe_m2 / basin_area_m2 if basin_area_m2 > 0 else 0.0
    pct_total = pct_aff + pct_ffe  # clamp if you ever need: min(100.0, pct_aff + pct_ffe)

    # IDs from filename and (optionally) attributes if present
    dem_i_file, dem_j_file = parse_ids(basin_path.stem)
    dem_i_attr = basin["dem_i"].iloc[0] if "dem_i" in basin.columns else None
    dem_j_attr = basin["dem_j"].iloc[0] if "dem_j" in basin.columns else None

    row = {
        "basin_file": basin_path.name,
        "dem_i_file": dem_i_file, "dem_j_file": dem_j_file,
        "dem_i_attr": dem_i_attr, "dem_j_attr": dem_j_attr,
        "basin_area_m2": basin_area_m2,
        "basin_area_km2": basin_area_km2,   # <-- added
        "afforestable_m2": aff_m2, "afforestable_km2": aff_km2, "afforestable_pct": pct_aff,
        "existing_forest_m2": ffe_m2, "existing_forest_km2": ffe_km2, "existing_forest_pct": pct_ffe,
        "total_future_forest_m2": total_future_forest_m2,
        "total_future_forest_km2": total_future_forest_km2,
        "total_future_forest_pct": pct_total,
        "points_used": len(pts_in),
    }

    # append and log
    pd.DataFrame([row]).to_csv(out_csv, mode="a", index=False, header=False)
    print(
        f"{basin_path.name} | basin {basin_area_km2:.3f} km² | "
        f"aff {aff_km2:.3f} km² ({pct_aff:.1f}%) | "
        f"forest {ffe_km2:.3f} km² ({pct_ffe:.1f}%) | "
        f"total {total_future_forest_km2:.3f} km² ({pct_total:.1f}%) | "
        f"points {len(pts_in)}"
    )
    sys.stdout.flush()

In [ ]:
landuse_df = pd.DataFrame({'Classify':['Open dry forest - Short', 'Quarry', 'Fields and Secondary Forest']})
proportion_forest = {
    'Open dry forest - Short': 1,
    'Quarry': 0,
    'Fields and Secondary Forest': 0.5,
    'Urban': 0,
}
proportion_afforestable = {
    'Open dry forest - Short': 0,
    'Quarry': 1,
    'Fields and Secondary Forest': 0.5,
    'Urban': 0,
}
landuse_df['forest_equivalent'] = landuse_df.Classify.map(proportion_forest)
landuse_df['afforestable'] = landuse_df.Classify.map(proportion_afforestable)
landuse_df

In [ ]:
# # ----------------------------------------------------------------------------
# # Apply the function to each row and "explode" the dictionary so each row has a single key-value pair.
# land_use_hydrobasins_intersection_saved['frac_dict'] = land_use_hydrobasins_intersection_saved.apply(Robyn_catchment_analysis.calculate_fractional_areas, axis=1)

# expanded_rows = []
# for idx, row in land_use_hydrobasins_intersection_saved.iterrows():
#     for key, value in row['frac_dict'].items():
#         new_row = row.copy()
#         new_row['LandUseCategory'] = key
#         new_row['Area'] = value
#         expanded_rows.append(new_row)
        
# expanded_gdf = gpd.GeoDataFrame(expanded_rows, crs=land_use_hydrobasins_intersection_saved.crs)
# # ----------------------------------------------------------------------------


In [ ]:

# # Calculate the forest flood equivalent area (i.e. the area that contributes to flood reduction)
# forest_flood_equivalent_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes']
# forest_flood_equivalent_area = (
#     forest_flood_equivalent_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'forest_flood_equivalent_area'})
# )

# # Calculate the afforestable area (including the agricultural component)
# afforestable_agri_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'afforestable_including_agriculture']
# afforestable_agri_area = (
#     afforestable_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'afforestable_including_agriculture_area'})
# )

# # Merge the results with the total catchment area DataFrame
# final_summary = total_catchment_area.merge(
#     forest_flood_equivalent_area, on='HYBAS_ID', how='left'
# ).merge(
#     afforestable_agri_area, on='HYBAS_ID', how='left'
# )

# # Replace NaN values with 0 if some HYBAS_IDs don't have one of the categories
# final_summary['forest_flood_equivalent_area'] = final_summary['forest_flood_equivalent_area'].fillna(0)
# final_summary['afforestable_including_agriculture_area'] = final_summary['afforestable_including_agriculture_area'].fillna(0)

# # Create a new column for the total future forest area including agriculture
# final_summary['total_future_forest_area_including_agri'] = (
#     final_summary['forest_flood_equivalent_area'] + final_summary['afforestable_including_agriculture_area']
# )

# # Calculate percentages relative to the total catchment area
# final_summary['forest_flood_equivalent_percentage'] = (
#     final_summary['forest_flood_equivalent_area'] / final_summary['total_catchment_area'] * 100
# ).round(2)
# final_summary['afforestable_including_agriculture_percentage'] = (
#     final_summary['afforestable_including_agriculture_area'] / final_summary['total_catchment_area'] * 100
# ).round(2)
# final_summary['total_future_forest_including_agri_percentage'] = (
#     final_summary['total_future_forest_area_including_agri'] / final_summary['total_catchment_area'] * 100
# ).round(2)

# # Reorder columns for clarity
# column_order = [
#     'HYBAS_ID', 
#     'total_catchment_area',
#     'forest_flood_equivalent_area',
#     'forest_flood_equivalent_percentage',
#     'afforestable_including_agriculture_area',
#     'afforestable_including_agriculture_percentage',
#     'total_future_forest_area_including_agri',
#     'total_future_forest_including_agri_percentage'
# ]
# final_summary = final_summary[column_order]

# # Display the final summary DataFrame
# display(final_summary)

# # Optional: Export the summary to CSV
# output_forests = output_path / "catchment_forest_summary_with_percentages.csv"
# final_summary.to_csv(output_forests, index=False)
# print(f"Data successfully exported to {output_path}")

In [ ]:
# # Step 1: Aggregate Area by Catchment and Land Use Category
# category_area = expanded_gdf.groupby(['HYBAS_ID', 'LandUseCategory'])['Area'].sum().reset_index()

# # Step 2: Calculate Total Area per Catchment from expanded_gdf
# total_area = expanded_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index().rename(columns={'Area': 'TotalArea'})

# # Step 3: Merge Aggregated Data with Total Area
# category_percentage = pd.merge(category_area, total_area, on='HYBAS_ID')

# # Step 4: Compute Percentage per Land Use Category
# category_percentage['Percentage'] = (category_percentage['Area'] / category_percentage['TotalArea']) * 100

# # (Optional) Step 5: Pivot Data for Easier Interpretation
# percentage_pivot = category_percentage.pivot(index='HYBAS_ID', columns='LandUseCategory', values='Percentage').fillna(0).reset_index()

# # Display the percentage DataFrame
# display("Percentage of Each Land Use Category within Each Catchment:")
# display(percentage_pivot.head())